# 강의 02 · 실습 2 — 구조화 출력 · (1) 강사 시연

## 1. 문제상황

- 팀의 회의록 담당자는 매주 금요일 회의 메모를 모델에게 넘겨 요약을 받습니다.
- 받은 요약은 사람이 읽기에는 좋지만, 담당자와 기한이 표 안의 글자로 섞여 있어 프로그램이 꺼내 쓸 칸이 없습니다.
- 같은 지시로 두 번 부르면 제목과 굵은 글씨의 위치가 바뀌어서, 첫 응답에 맞춰 만든 잘라내기 코드가 두 번째 응답에서는 아무것도 찾지 못합니다.
- 담당자별로 할 일을 보내는 다음 단계는 사람이 요약을 다시 읽고 손으로 옮겨 적어야 합니다.

## 2. 문제와 목표

- **문제**: 모델의 답이 자유 텍스트라서 모양이 호출마다 달라지고, 뒤 단계의 프로그램이 답에서 값을 꺼낼 수 없습니다.
- **목표**: 받고 싶은 출력의 모양을 클래스로 선언하고, 그 모양을 지정해 모델을 호출하고, 돌아온 답을 파이썬 객체로 받아 담당자별 메시지를 만드는 처리 흐름을 만듭니다. 어긋난 답이 어디서 걸리는지도 확인합니다.
    - 출력의 모양: 결정사항 리스트, 할 일 리스트(할 일마다 일·담당자·기한), 미결 리스트.
    - 어긋난 답: 리스트여야 하는 필드에 문자열, 필수 필드 누락, 달력에 없는 날짜(예: 2026-13-45).
- **목표 달성 여부의 판정 기준**: 회의 메모를 넣었을 때 결정사항·할 일·미결이 선언한 클래스의 인스턴스로 돌아오고, 할 일이 담당자별 메시지로 묶여 출력되며, 일부러 어긋난 입력을 넣었을 때 검증 오류가 나는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec02_ex02_s1_diagram.svg)

## 4. 단계별 요구사항

1. **출력 스키마를 선언합니다.**
    - 할 일 한 개를 나타내는 `ActionItem`(`task`·`owner`·`due`)과 회의 요약 전체를 나타내는 `MeetingSummary`(`decisions`·`action_items`·`open_questions`)를 `BaseModel`을 상속한 클래스로 선언합니다.
    - 기한은 없을 수 있으므로 `str | None`으로 둡니다.
2. **값 검증기를 답니다.**
    - `ActionItem`의 `due`에 검증기를 붙여, `YYYY-MM-DD` 형식으로 적힌 값이 달력에 있는 날짜인지 확인합니다.
    - 달력에 없는 날짜면 검증이 실패합니다.
3. **스키마를 지정해 모델을 호출합니다.**
    - 시스템 프롬프트에 추출 규칙(담당자가 없는 할 일의 `owner`는 '미정', 메모에 없는 내용은 지어내지 않는다)을 적고, `response_format`에 `MeetingSummary`를 지정해 모델을 한 번 호출합니다.
    - 돌아온 답은 JSON 문자열입니다.
4. **돌아온 문자열을 객체로 되돌리고 다음 단계로 넘깁니다.**
    - 응답 문자열을 `MeetingSummary.model_validate_json`으로 파싱해 인스턴스로 만들고, 할 일 목록을 담당자별로 묶어 보낼 메시지를 만듭니다.
5. **검증 실패를 관찰합니다.**
    - 리스트여야 하는 필드에 문자열을 넣고 필수 필드를 뺀 딕셔너리를 `model_validate`에 넣어 `ValidationError`를 관찰합니다.
    - 같은 어긋난 날짜를 값 검증기가 없는 클래스와 있는 클래스에 각각 넣어, 모양 검사와 값 검증의 차이를 봅니다.

## 5. 코드 골격 — 구조화 출력(pydantic) 4단

파이댄틱(pydantic)으로 구조화 출력을 받는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 스키마 선언 | 받아 낼 출력의 모양을 클래스로 선언하고 값 검증기를 답니다 | `class MeetingSummary(BaseModel)`, `@field_validator` | 1, 2 |
| ② 스키마를 건 호출 | 모델 호출에 스키마를 지정해 출력 형식을 강제합니다 | `completion(..., response_format=MeetingSummary)` | 3 |
| ③ 객체 수신·파싱 | 돌아온 문자열을 클래스로 되돌려 파이썬 객체로 씁니다 | `MeetingSummary.model_validate_json(...)` | 4 |
| ④ 검증 실패 관찰 | 일부러 어긋난 값을 넣어 어디서 어떻게 걸리는지 봅니다 | `ValidationError`, `model_validate(broken)` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import json
import os
import re
from datetime import date

from dotenv import load_dotenv, find_dotenv
from litellm import completion
from pydantic import BaseModel, ValidationError, field_validator

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
print("준비를 마쳤습니다.")

### 단계 ① — 스키마 선언 (요구사항 1, 2)

- 출력의 필드 이름과 타입을 클래스로 선언합니다. 선언한 모양은 JSON Schema로 바뀌어 모델에 전달됩니다.
- 선언에 없는 내용은 답에 담기지 않습니다. 없을 수 있는 값은 `str | None`으로 적어 두어야 「없음」을 적을 수 있습니다.
- `@field_validator`는 모양 검사가 통과시킨 값이 실제로 옳은지 보는 값 검증기입니다. 타입이 맞는 문자열이라도 달력에 없는 날짜면 걸립니다.

In [ ]:
class ActionItem(BaseModel):
    task: str          # 할 일
    owner: str         # 담당자 (회의에서 지정이 없으면 "미정")
    due: str | None    # 기한 (없으면 None)

    @field_validator("due")
    @classmethod
    def due_must_exist_on_calendar(cls, v):
        """YYYY-MM-DD 형식으로 적힌 기한이 달력에 있는 날짜인지 확인한다."""
        if v and re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
            date.fromisoformat(v)  # 2026-13-45처럼 달력에 없는 날짜면 ValueError
        return v


class MeetingSummary(BaseModel):
    decisions: list[str]            # 결정사항
    action_items: list[ActionItem]  # 할 일 목록
    open_questions: list[str]       # 미결


print("스키마의 필드:", list(MeetingSummary.model_fields))

### 단계 ② — 스키마를 건 호출 (요구사항 3)

- 시스템 프롬프트에는 추출 규칙을 적습니다. 회의 때마다 바뀌는 입력은 메모뿐이고, 규칙과 스키마는 그대로 둡니다.
- `response_format`에 스키마 클래스를 넘기면 모델은 그 모양에 맞는 JSON 문자열만 돌려줍니다.
- 이 셀은 모델을 한 번 호출합니다.

In [ ]:
SYSTEM = ("회의 메모에서 결정사항, 할 일, 미결 사항을 추출한다. "
          "담당자가 없는 할 일의 owner는 '미정'으로 적는다. "
          "메모에 없는 내용을 지어내지 않는다.")

MEETING_MEMO = """주간 회의 메모 (금요일)
- 다음 배포는 수요일로 확정했다.
- 접속 오류 재현 조건을 민수 씨가 다음 회의 전까지 정리하기로 했다.
- 매뉴얼 개편은 다다음 주에 다시 논의한다.
- 신규 문의 응대 템플릿 초안은 지현 씨가 화요일까지 만들기로 했다.
- 서버 증설 예산은 결론을 내지 못했다."""


def call_with_schema(memo: str) -> str:
    """스키마를 지정해 모델을 한 번 호출하고, 돌아온 JSON 문자열을 돌려준다."""
    res = completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": memo},
        ],
        response_format=MeetingSummary,
    )
    return res.choices[0].message.content


raw = call_with_schema(MEETING_MEMO)
print("타입:", type(raw).__name__)
print(raw)

### 단계 ③ — 객체 수신·파싱 (요구사항 4)

- 돌아온 답은 아직 문자열입니다. `model_validate_json`이 문자열을 스키마 클래스의 인스턴스로 되돌리면서 모양 검사와 값 검증을 함께 합니다.
- 인스턴스는 변수 하나에 담겨 다음 단계로 넘어갑니다. 이 실습에서는 할 일을 담당자별 메시지로 묶습니다.

In [ ]:
summary = MeetingSummary.model_validate_json(raw)
print("타입:", type(summary).__name__, "/ 할 일 수:", len(summary.action_items))
print(json.dumps(summary.model_dump(), ensure_ascii=False, indent=2))


def notify_owners(summary: MeetingSummary) -> dict:
    """할 일을 담당자별로 묶어 보낼 메시지를 만든다."""
    out = {}
    for item in summary.action_items:
        due = f" (기한: {item.due})" if item.due else ""
        head = f"{item.owner} 님, 이번 회의에서 정해진 할 일입니다."
        out[item.owner] = out.get(item.owner, head) + f"\n- {item.task}{due}"
    return out


print()
for owner, msg in notify_owners(summary).items():
    print(f"[{owner}]")
    print(msg)
    print()

### 단계 ④ — 검증 실패 관찰 (요구사항 5)

- 모델을 부르지 않습니다. 손으로 만든 어긋난 딕셔너리를 `model_validate`에 넣어 검증이 어디서 걸리는지 봅니다.
- 첫 번째 입력은 리스트여야 하는 필드에 문자열이 있고 필수 필드 `owner`와 `due`가 빠져 있습니다. 두 번째 입력은 타입은 맞지만 달력에 없는 날짜입니다.
- 값 검증기가 없는 클래스는 두 번째 입력을 통과시키고, 값 검증기가 있는 클래스는 걸러 냅니다.

In [ ]:
broken = {
    "decisions": "배포는 수요일",                   # 리스트여야 하는데 문자열
    "action_items": [{"task": "재현 조건 정리"}],  # owner 누락
    "open_questions": [],
}
try:
    MeetingSummary.model_validate(broken)
except ValidationError as e:
    print("=== 검증 실패의 모양 (ValidationError) ===")
    print(e)


class ActionItemShapeOnly(BaseModel):
    """값 검증기가 없는 모양 검사기. 타입과 필수 필드만 본다."""
    task: str
    owner: str
    due: str | None


broken_item = {"task": "접속 오류 재현 조건 정리", "owner": "민수", "due": "2026-13-45"}
print()
print("=== 모양 검사만 (값 검증기 없음) ===")
print(ActionItemShapeOnly.model_validate(broken_item))
try:
    ActionItem.model_validate(broken_item)
except ValidationError as e:
    print()
    print("=== 모양 검사 + 값 검증기 ===")
    print(e)

## 7. 실행 결과 확인

위 실행 기록에서 다음 네 가지를 확인합니다.

1. 단계 ②의 출력은 `str`이며, 첫 글자가 여는 중괄호인 JSON 문자열입니다. 인사말이나 설명 문장이 붙어 있지 않습니다.
2. 단계 ③의 타입 줄에 `MeetingSummary`가 찍히고, 할 일 수가 2입니다. 담당자별 메시지가 민수 씨와 지현 씨의 이름으로 묶여 찍힙니다.
3. 단계 ④의 첫 번째 `ValidationError`에 `decisions`, `action_items.0.owner`, `action_items.0.due` 세 필드가 함께 찍힙니다. `due`는 `None`일 수 있어도 필드 자체는 있어야 합니다. 검증은 첫 오류에서 멈추지 않고 어긋난 자리를 전부 모아 보고합니다.
4. 단계 ④에서 같은 `2026-13-45`를 넣었을 때, 모양 검사만 하는 클래스는 인스턴스를 만들고, 값 검증기가 있는 클래스는 `due` 필드에서 `ValidationError`를 냅니다.